In [2]:
import pandas as pd

df=pd.read_csv("Car details v3.csv")
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [3]:
df=df.drop(columns=['name','torque'],axis=1)
df.head()

,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats
0,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,5.0
1,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,5.0
2,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,5.0
3,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,5.0
4,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,5.0


In [4]:
df['mileage']=df['mileage'].str.extract(r"([\d\.]+)").astype(float)
df['engine']=df['engine'].str.extract(r"([\d\.]+)").astype(float)
df['max_power']=df['max_power'].str.extract(r"([\d\.]+)").astype(float)

In [5]:
df.isna().sum()

year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          221
engine           221
max_power        216
seats            221
dtype: int64

In [6]:
df=df.fillna(df.median(numeric_only=True))

In [7]:
df=pd.get_dummies(df,columns=['fuel','seller_type','transmission','owner'],drop_first=True)
df.tail()

,year,selling_price,km_driven,mileage,engine,max_power,seats,fuel_Diesel,fuel_LPG,fuel_Petrol,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Manual,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
8123,2013,320000,110000,18.50,1197.0,82.85,5.0,False,False,True,True,False,True,False,False,False,False
8124,2007,135000,119000,16.80,1493.0,110.00,5.0,True,False,False,True,False,True,True,False,False,False
8125,2009,382000,120000,19.30,1248.0,73.90,5.0,True,False,False,True,False,True,False,False,False,False
8126,2013,290000,25000,23.57,1396.0,70.00,5.0,True,False,False,True,False,True,False,False,False,False
8127,2013,290000,25000,23.57,1396.0,70.00,5.0,True,False,False,True,False,True,False,False,False,False


In [8]:
df.isna().sum()

year                            0
selling_price                   0
km_driven                       0
mileage                         0
engine                          0
max_power                       0
seats                           0
fuel_Diesel                     0
fuel_LPG                        0
fuel_Petrol                     0
seller_type_Individual          0
seller_type_Trustmark Dealer    0
transmission_Manual             0
owner_Fourth & Above Owner      0
owner_Second Owner              0
owner_Test Drive Car            0
owner_Third Owner               0
dtype: int64

In [9]:
X=df.drop('selling_price',axis=1)
y=df['selling_price']
print(X.shape,y.shape)

(8128, 16) (8128,)


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [15]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score,mean_squared_error

dt=DecisionTreeRegressor(random_state=42)
dt.fit(X_train,y_train)

dt_pred=dt.predict(X_test)

print("R2 Score =",r2_score(y_test,dt_pred))
print("MSE =",mean_squared_error(y_test,dt_pred),"In Ruppee =",mean_squared_error(y_test,dt_pred)**0.5)

R2 Score = 0.9566054321762238
MSE = 28444464006.570408 In Ruppee = 168654.8665368729


In [16]:
from sklearn.ensemble import RandomForestRegressor

rf=RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train,y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [17]:
rf_pred=rf.predict(X_test)

print("R2 Score =",r2_score(y_test,rf_pred))
print("MSE =",mean_squared_error(y_test,rf_pred),"In Ruppee =",mean_squared_error(y_test,rf_pred)**0.5)

R2 Score = 0.9688673758206463
MSE = 20406950738.532566 In Ruppee = 142852.8989503978


In [19]:
importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

In [20]:
importance

max_power                       0.724393
year                            0.191500
km_driven                       0.027372
mileage                         0.019352
engine                          0.019288
seller_type_Individual          0.006648
transmission_Manual             0.002675
seats                           0.002620
fuel_Petrol                     0.001987
fuel_Diesel                     0.001896
owner_Second Owner              0.001414
owner_Test Drive Car            0.000421
owner_Third Owner               0.000232
owner_Fourth & Above Owner      0.000114
seller_type_Trustmark Dealer    0.000080
fuel_LPG                        0.000006
dtype: float64